# Lekcja 2.6 — eksperymenty: iteracyjne pogłębianie

Uruchamiaj komórki po kolei. Przed każdym eksperymentem zapisz swoje przewidywanie w komórce „Twoje przewidywanie” (kliknij ją dwukrotnie, żeby edytować), a dopiero potem uruchom kod.

Pierwsza komórka wybiera implementację:

- `"mine"` używa Twojego `task.py`; najpierw rozwiąż część B zadania;
- `"reference"` używa implementacji wzorcowej kursu. Wpisz `IMPLEMENTATION = "reference"`, jeśli chcesz uruchomić notebook przed skończeniem zadania.

In [ ]:
%load_ext autoreload
%autoreload 2
import os

# "mine" uses your task.py; "reference" uses the course's reference implementation.
IMPLEMENTATION = os.environ.get("INTROAI_IMPLEMENTATION", "mine")
if IMPLEMENTATION == "mine":
    import task as algorithms
else:
    from introai.search import reference as algorithms

Funkcja `run` uruchamia jedno przeszukiwanie i zwraca węzeł celu oraz liczbę rozwinięć. Funkcja `largest_frontier` mierzy największy frontier; Twój kod tego nie zgłasza, więc pamięć mierzymy zawsze na wersjach wzorcowych kursu. BFS pochodzi z wersji wzorcowej (lekcja 2.2). Funkcja `tree(b, d)` buduje pełne drzewo o współczynniku rozgałęzienia `b` i głębokości `d`, z celem w najgłębszym liściu po prawej stronie, czyli w miejscu, do którego DFS dociera na końcu.

In [ ]:
from pathlib import Path

import matplotlib.pyplot as plt

from introai.search import CUTOFF, reference
from introai.search.examples import complete_tree
from introai.search.problems import GraphProblem
from introai.testing.search import max_frontier
from introai.trace import TraceLimitExceeded, record
from introai.world.warehouse import RouteProblem, WarehouseMap

MAPS = Path.cwd().parents[2] / "data" / "maps"
HINT = 'check task.py (or set IMPLEMENTATION = "reference" in the first cell)'


def run(search, *args, limit=200_000):
    """Run one search; return its result and the number of expansions."""
    try:
        with record(limit=limit, kinds={"expand"}) as trace:
            result = search(*args)
    except NotImplementedError:
        raise RuntimeError(f"{search.__name__} is not written yet - {HINT}") from None
    except (TraceLimitExceeded, RecursionError):
        raise RuntimeError(f"{search.__name__} expanded more than {limit} nodes - {HINT}") from None
    return result, len(trace.of_kind("expand"))


def largest_frontier(search, problem):
    with record(kinds={"frontier"}) as trace:
        search(problem)
    return max_frontier(trace)


def tree(b, d):
    adjacency = complete_tree(b, d).adjacency
    return GraphProblem(adjacency, "r", "r" + str(b - 1) * d)

## Eksperyment 1: pełne drzewa

Porównujemy BFS i IDS na pełnych drzewach o współczynniku rozgałęzienia 2, 3 i 4 i głębokości od 2 do 8.

**Twoje przewidywanie.** Ile razy więcej węzłów rozwinie IDS niż BFS przy *b* = 2, a ile przy *b* = 4? Ile węzłów naraz trzyma we frontier każdy z nich przy *b* = 3 i *d* = 8?

*Moje przewidywanie:* …

In [ ]:
depths = range(2, 9)
ratios = {}
memory = {"BFS": [], "IDS": []}
print(f"{'b':>2} {'d':>2} {'BFS':>7} {'IDS':>7} {'IDS/BFS':>8} {'mem BFS':>8} {'mem IDS':>8}")
for b in (2, 3, 4):
    ratios[b] = []
    for d in depths:
        _, bfs_work = run(reference.bfs, tree(b, d))
        found, ids_work = run(algorithms.iterative_deepening_search, tree(b, d))
        if found is None or found == CUTOFF:
            raise RuntimeError(
                f"iterative deepening did not find the goal of tree({b}, {d}) - {HINT}"
            )
        ratios[b].append(ids_work / bfs_work)
        bfs_memory = largest_frontier(reference.bfs, tree(b, d))
        ids_memory = largest_frontier(reference.iterative_deepening_search, tree(b, d))
        if b == 3:
            memory["BFS"].append(bfs_memory)
            memory["IDS"].append(ids_memory)
        print(
            f"{b:>2} {d:>2} {bfs_work:>7} {ids_work:>7} {ids_work / bfs_work:>8.2f} "
            f"{bfs_memory:>8} {ids_memory:>8}"
        )

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13, 4))
ax = axes[0]
for b, color in zip((2, 3, 4), ("#2a78d6", "#eb6834", "#1baf7a")):
    ax.plot(
        list(depths),
        ratios[b],
        color=color,
        linewidth=2,
        marker="o",
        markersize=8,
        markeredgecolor="#fcfcfb",
        markeredgewidth=1.5,
        label=f"b = {b}",
    )
    ax.annotate(
        f"b = {b}",
        (depths[-1], ratios[b][-1]),
        textcoords="offset points",
        xytext=(8, 0),
        va="center",
        color="#0b0b0b",
    )
ax.set_title("Expansions of IDS divided by expansions of BFS", loc="left", color="#0b0b0b")
ax.set_xlabel("depth of the goal", color="#52514e")
ax.margins(x=0.12)
ax.legend(frameon=False, loc="upper left", labelcolor="#0b0b0b")
ax.grid(axis="y", color="#e1e0d9", linewidth=0.8)
ax.set_axisbelow(True)
for side in ("top", "right"):
    ax.spines[side].set_visible(False)
for side in ("left", "bottom"):
    ax.spines[side].set_color("#c3c2b7")
ax.tick_params(colors="#52514e")

ax = axes[1]
for label, color in (("BFS", "#2a78d6"), ("IDS", "#eb6834")):
    ax.plot(
        list(depths),
        memory[label],
        color=color,
        linewidth=2,
        marker="o",
        markersize=8,
        markeredgecolor="#fcfcfb",
        markeredgewidth=1.5,
        label=label,
    )
ax.set_yscale("log")
ax.set_title("Largest frontier for b = 3 (log scale)", loc="left", color="#0b0b0b")
ax.set_xlabel("depth of the goal", color="#52514e")
ax.legend(frameon=False, loc="upper left", labelcolor="#0b0b0b")
ax.grid(axis="y", color="#e1e0d9", linewidth=0.8)
ax.set_axisbelow(True)
for side in ("top", "right"):
    ax.spines[side].set_visible(False)
for side in ("left", "bottom"):
    ax.spines[side].set_color("#c3c2b7")
ax.tick_params(colors="#52514e")
fig.tight_layout()

**Do przemyślenia.** Do jakich wartości zbliżają się stosunki pracy dla *b* = 2, 3 i 4? Porównaj je ze wzorem *b*/(*b* − 1). Jak rośnie frontier BFS, a jak frontier IDS, gdy głębokość rośnie o 1?

## Eksperyment 2: magazyn

Teraz te same algorytmy na mapach magazynu, gdzie każdy krok kosztuje 1, a do wielu pól prowadzi bardzo wiele dróg.

**Twoje przewidywanie.** Czy IDS rozwinie na mapie z alejkami około dwa razy więcej pól niż BFS, jak na drzewie? A na otwartej podłodze?

*Moje przewidywanie:* …

In [ ]:
names = ["small", "aisles", "open_floor", "depot"]
per_limit = {}
for name in names:
    warehouse = WarehouseMap.from_file(MAPS / f"{name}.txt")
    bfs_node, bfs_work = run(reference.bfs, RouteProblem(warehouse))
    counts, limit = [], 0
    while True:
        result, work = run(algorithms.depth_limited_search, RouteProblem(warehouse), limit)
        counts.append(work)
        if result != CUTOFF:
            break
        limit += 1
    per_limit[name] = counts
    print(
        f"{name:>10}: route {len(bfs_node.solution())} steps, BFS expanded {bfs_work}, "
        f"IDS expanded {sum(counts)} in {len(counts)} rounds"
    )

fig, ax = plt.subplots(figsize=(8, 4))
for name, color in zip(("aisles", "open_floor"), ("#2a78d6", "#eb6834")):
    counts = per_limit[name]
    ax.plot(
        range(1, len(counts)),
        counts[1:],  # limit 0 expands nothing, which a log scale cannot show
        color=color,
        linewidth=2,
        marker="o",
        markersize=8,
        markeredgecolor="#fcfcfb",
        markeredgewidth=1.5,
        label=name,
    )
ax.set_yscale("log")
ax.set_title("Nodes expanded by depth-limited search, by limit", loc="left", color="#0b0b0b")
ax.set_xlabel("depth limit", color="#52514e")
ax.set_ylabel("nodes expanded (log scale)", color="#52514e")
ax.legend(frameon=False, loc="upper left", labelcolor="#0b0b0b")
ax.grid(axis="y", color="#e1e0d9", linewidth=0.8)
ax.set_axisbelow(True)
for side in ("top", "right"):
    ax.spines[side].set_visible(False)
for side in ("left", "bottom"):
    ax.spines[side].set_color("#c3c2b7")
ax.tick_params(colors="#52514e")
fig.tight_layout()

**Do przemyślenia.** Dlaczego na mapach IDS przegrywa z BFS dużo wyraźniej niż na drzewie? Przypomnij sobie lekcję 2.1: co IDS pamięta, a czego nie, i ile różnych dróg prowadzi do jednego pola na otwartej podłodze. Kiedy w takim razie warto używać IDS?

## Podsumowanie

Wróć do pytań kontrolnych w lekcji. Po własnej próbie możesz porównać swoje wyniki i wnioski z odpowiedziami w `solutions/02_search/06_iterative_deepening/answers.pl.md`.